# 🎯 Hybrid ATS Resume Analyzer
### TF-IDF + Semantic Similarity (MiniLM) + Section-Aware Scoring + Manipulation Detection

**What this notebook does:**
- Upload a **PDF or DOCX** resume
- Scores it against all jobs in the dataset using a **hybrid model**
- Shows **ATS score**, **top matching jobs**, **matched skills**, **missing skills**, and **recommendations**

**Formula:** `Final Score = (0.4 × TF-IDF + 0.6 × Semantic) × 100 − Manipulation Penalty`


## ⚙️ Step 1 — Install Dependencies
> Run this once per Colab session.

In [20]:
!pip install -q sentence-transformers
!pip install -q python-docx
!pip install -q PyMuPDF
!pip install -q scikit-learn pandas numpy
print("✅ All packages installed successfully")

✅ All packages installed successfully


## 📦 Step 2 — Import Libraries

In [21]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display, HTML

# PDF parsing
import fitz  # PyMuPDF

# DOCX parsing
from docx import Document

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Semantic Embeddings
from sentence_transformers import SentenceTransformer

# Google Colab file upload
from google.colab import files

warnings.filterwarnings("ignore")
print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## 📂 Step 3 — Upload Dataset
> Upload `resume_analyzer_skills_dataset.csv` when prompted.

In [37]:
print("Please upload your dataset CSV file: resume_analyzer_skills_dataset.csv")
uploaded_ds = files.upload()
dataset_filename = list(uploaded_ds.keys())[0]
print(f"✅ Dataset uploaded: {dataset_filename}")

Please upload your dataset CSV file: resume_analyzer_skills_dataset.csv


Saving resume_analyzer_skills_dataset.csv to resume_analyzer_skills_dataset (3).csv
✅ Dataset uploaded: resume_analyzer_skills_dataset (3).csv


## 🗄️ Step 4 — Load and Preview Dataset

In [38]:
df = pd.read_csv(dataset_filename)

# Clean up skills column
df["Skills_Required"] = df["Skills_Required"].str.lower().str.strip()

# Parse skills into a list per row
df["Skills_List"] = df["Skills_Required"].apply(
    lambda x: [s.strip() for s in x.split(",")]
)

print(f"✅ Dataset loaded: {len(df)} job entries")
print(f"📌 Unique Job Titles: {df['Job_Title'].nunique()}")
print("\n--- Job Categories ---")
print(df['Job_Title'].value_counts().to_string())
print("\n--- Sample Rows ---")
display(df[["Job_ID", "Job_Title", "Experience_Level", "Min_Experience_Years", "Skill_Count"]].head(10))

✅ Dataset loaded: 74 job entries
📌 Unique Job Titles: 74

--- Job Categories ---
Job_Title
Software Engineer                       1
Frontend Developer                      1
Backend Developer                       1
Full Stack Developer                    1
Data Scientist                          1
Data Analyst                            1
Machine Learning Engineer               1
DevOps Engineer                         1
Cloud Engineer                          1
Cybersecurity Analyst                   1
Database Administrator                  1
Mobile Developer                        1
QA Engineer                             1
UI/UX Designer                          1
Product Manager                         1
Project Manager                         1
Site Reliability Engineer               1
AI Engineer                             1
Embedded Systems Engineer               1
Data Engineer                           1
Blockchain Developer                    1
Game Developer             

,Job_ID,Job_Title,Experience_Level,Min_Experience_Years,Skill_Count
0,Software_Engineer,Software Engineer,All Levels,0,28
1,Frontend_Developer,Frontend Developer,All Levels,0,26
2,Backend_Developer,Backend Developer,All Levels,0,26
3,Full_Stack_Developer,Full Stack Developer,All Levels,0,26
4,Data_Scientist,Data Scientist,All Levels,0,26
5,Data_Analyst,Data Analyst,All Levels,0,24
6,Machine_Learning_Engineer,Machine Learning Engineer,All Levels,0,25
7,DevOps_Engineer,DevOps Engineer,All Levels,0,25
8,Cloud_Engineer,Cloud Engineer,All Levels,0,25
9,Cybersecurity_Analyst,Cybersecurity Analyst,All Levels,0,24


## 📄 Step 5 — Resume Parser (PDF / DOCX)
> Extracts full text and splits resume into sections (Skills, Experience, Projects, Education).

In [39]:
# Section header keywords
SECTION_KEYWORDS = {
    "skills":     ["skills", "technical skills", "core competencies", "key skills", "technologies", "tools", "competencies"],
    "experience": ["experience", "work experience", "employment", "work history", "professional experience", "career history"],
    "projects":   ["projects", "project experience", "personal projects", "academic projects", "key projects"],
    "education":  ["education", "academic background", "qualifications", "certifications", "training", "academics"]
}

def extract_text_from_pdf(path):
    """Extract raw text from a PDF file using PyMuPDF."""
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def extract_text_from_docx(path):
    """Extract raw text from a DOCX file."""
    doc = Document(path)
    return "\n".join([para.text for para in doc.paragraphs])

def extract_resume_text(path):
    """Auto-detect file type and extract text."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return extract_text_from_pdf(path)
    elif ext in [".docx", ".doc"]:
        return extract_text_from_docx(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}. Please upload PDF or DOCX.")

def parse_resume_sections(text):
    """
    Parse resume into sections: skills, experience, projects, education, other.
    Returns a dict mapping section_name -> list of text lines.
    """
    lines = text.split("\n")
    sections = {k: [] for k in SECTION_KEYWORDS}
    sections["other"] = []
    current_section = "other"

    for line in lines:
        line_clean = line.strip().lower()
        if not line_clean:
            continue
        # Check if line is a section header (short line matching a keyword)
        matched_section = None
        if len(line_clean) < 60:  # section headers are usually short
            for section, keywords in SECTION_KEYWORDS.items():
                for kw in keywords:
                    if re.search(rf"\b{re.escape(kw)}\b", line_clean):
                        matched_section = section
                        break
                if matched_section:
                    break
        if matched_section:
            current_section = matched_section
        else:
            sections[current_section].append(line.strip())

    return sections

def sections_to_weighted_text(sections):
    """
    Return weighted resume text where more important sections get higher repetition.
    Weights: Experience=40%, Skills=30%, Projects=20%, Education=10%
    """
    WEIGHTS = {"skills": 3, "experience": 4, "projects": 2, "education": 1, "other": 1}
    weighted_parts = []
    for section, weight in WEIGHTS.items():
        section_text = " ".join(sections.get(section, []))
        if section_text.strip():
            weighted_parts.extend([section_text] * weight)
    return " ".join(weighted_parts)

print("✅ Resume parser functions defined")

✅ Resume parser functions defined


## 🔤 Step 6 — Text Preprocessing + Abbreviation Expansion
> Expands technical abbreviations (NLP → natural language processing, ML → machine learning, etc.)

In [40]:
# Abbreviation map: regex pattern → expansion
ABBREVIATION_MAP = {
    r"\bnlp\b": "natural language processing",
    r"\bml\b": "machine learning",
    r"\bai\b": "artificial intelligence",
    r"\bdl\b": "deep learning",
    r"\bcv\b": "computer vision",
    r"\bjs\b": "javascript",
    r"\bts\b": "typescript",
    r"\bapi\b": "application programming interface",
    r"\bui\b": "user interface",
    r"\bux\b": "user experience",
    r"\bci/cd\b": "continuous integration continuous deployment",
    r"\bllm\b": "large language model",
    r"\bsql\b": "structured query language",
    r"\bhtml\b": "hypertext markup language",
    r"\bcss\b": "cascading style sheets",
    r"\boop\b": "object oriented programming",
    r"\baws\b": "amazon web services",
    r"\bgcp\b": "google cloud platform",
    r"\bds\b": "data science",
    r"\bda\b": "data analysis",
    r"\brpa\b": "robotic process automation",
    r"\biot\b": "internet of things",
    r"\bpoc\b": "proof of concept",
    r"\brest\b": "representational state transfer api",
    r"\bvm\b": "virtual machine",
    r"\bci\b": "continuous integration",
    r"\bcd\b": "continuous deployment",
}

def expand_abbreviations(text):
    """Expand known abbreviations in text."""
    text_lower = text.lower()
    for pattern, expansion in ABBREVIATION_MAP.items():
        text_lower = re.sub(pattern, expansion, text_lower)
    return text_lower

def preprocess_text(text):
    """Full preprocessing: lowercase, expand abbreviations, clean special chars."""
    text = text.lower()
    text = expand_abbreviations(text)
    text = re.sub(r"[^a-z0-9\s\/\+\#\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("✅ Preprocessing and abbreviation expansion defined")
print("Sample expansion:", expand_abbreviations("I have experience in NLP, ML, and AWS."))

✅ Preprocessing and abbreviation expansion defined
Sample expansion: i have experience in natural language processing, machine learning, and amazon web services.


## 🚨 Step 7 — Keyword Stuffing / Manipulation Detector
> Detects abnormal keyword repetition and applies score penalties.

In [41]:
def detect_manipulation(text, threshold=5, density_threshold=0.07):
    """
    Detect keyword stuffing in resume text.

    Args:
        text: preprocessed resume text
        threshold: max allowed repetitions of any meaningful word (>3 chars)
        density_threshold: max allowed ratio of a word's frequency to total words

    Returns:
        (is_manipulated: bool, penalty: float 0-20, flags: list of str)
    """
    # Only count meaningful words (>3 chars, not stopwords)
    stopwords = {"with", "that", "this", "have", "from", "they", "been",
                 "were", "will", "your", "more", "also", "some", "than",
                 "into", "over", "such", "when", "would", "could", "which"}
    words = [w for w in re.findall(r"\b[a-z]{4,}\b", text.lower())
             if w not in stopwords]

    if not words:
        return False, 0.0, []

    word_counts = Counter(words)
    total_words = len(words)
    flags = []
    penalty = 0.0

    for word, count in word_counts.most_common(20):
        density = count / total_words
        if count > threshold and density > density_threshold:
            flags.append(f"'{word}' appears {count} times (density: {density:.1%})")
            excess = count - threshold
            penalty += min(excess * 1.5, 8.0)  # cap per-word penalty at 8

    total_penalty = round(min(penalty, 20.0), 2)  # max penalty = 20
    return len(flags) > 0, total_penalty, flags

print("✅ Manipulation detector defined")

✅ Manipulation detector defined


## 📊 Step 8 — TF-IDF Scorer
> Uses `sublinear_tf=True` to reduce keyword stuffing and `ngram_range=(1,2)` to capture phrases like 'machine learning'.

In [42]:
def compute_tfidf_score(resume_text, job_skills_text):
    """
    Compute TF-IDF cosine similarity between resume and job skills.
    Returns similarity score between 0 and 1.
    """
    vectorizer = TfidfVectorizer(
        sublinear_tf=True,       # log(1+tf) to reduce stuffing impact
        ngram_range=(1, 2),      # capture unigrams + bigrams
        stop_words="english",
        max_features=8000
    )
    try:
        tfidf_matrix = vectorizer.fit_transform([resume_text, job_skills_text])
        score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return float(score)
    except Exception:
        return 0.0

print("✅ TF-IDF scorer defined")

✅ TF-IDF scorer defined


## 🧠 Step 9 — Semantic Similarity Scorer (MiniLM)
> Loads `all-MiniLM-L6-v2` — lightweight, free, runs locally. Understands meaning, not just keywords.

In [43]:
print("Loading sentence-transformer model: all-MiniLM-L6-v2 ...")
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Semantic model loaded successfully")

def compute_semantic_score(text_a, text_b):
    """
    Compute semantic similarity between two texts using MiniLM embeddings.
    Returns cosine similarity between 0 and 1.
    """
    try:
        # Truncate long texts to avoid slow encoding
        text_a = text_a[:3000]
        text_b = text_b[:2000]
        embeddings = semantic_model.encode([text_a, text_b], convert_to_numpy=True, show_progress_bar=False)
        a, b = embeddings[0], embeddings[1]
        score = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
        return max(0.0, score)
    except Exception as e:
        print(f"Semantic scoring error: {e}")
        return 0.0

print("✅ Semantic scorer defined")

Loading sentence-transformer model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Semantic model loaded successfully
✅ Semantic scorer defined


## 🔍 Step 10 — Skill Matching Engine
> Matches job-required skills against resume using exact and expanded-form matching.

In [44]:
def match_skills(resume_text_raw, job_skills_list):
    """
    Match skills from job requirements against resume text.
    Handles both exact and abbreviation-expanded forms.
    Returns: (matched_skills: list, missing_skills: list)
    """
    resume_lower = resume_text_raw.lower()
    resume_expanded = expand_abbreviations(resume_lower)

    matched = []
    missing = []

    for skill in job_skills_list:
        skill_clean = skill.strip().lower()
        skill_expanded = expand_abbreviations(skill_clean)

        # Try all matching forms
        found = (
            skill_clean in resume_expanded
            or skill_expanded in resume_expanded
            or re.search(rf"\b{re.escape(skill_clean)}\b", resume_lower) is not None
        )

        if found:
            matched.append(skill)
        else:
            missing.append(skill)

    return matched, missing

print("✅ Skill matching engine defined")

✅ Skill matching engine defined


## ⚡ Step 11 — Hybrid Scorer + Job Matching

> **Formula:** `Final Score = (0.4 × TF-IDF + 0.6 × Semantic) × 100 − Manipulation Penalty`

| Component | Weight | Purpose |
|-----------|--------|---------|
| TF-IDF | 40% | Exact keyword precision |
| Semantic | 60% | Contextual / meaning-based similarity |
| Penalty | -0 to -20 | Punishes keyword stuffing |


In [45]:
from collections import defaultdict

def compute_hybrid_score(tfidf_score, semantic_score):
    """Weighted hybrid: 40% TF-IDF + 60% Semantic."""
    return (0.4 * tfidf_score + 0.6 * semantic_score) * 100

def analyze_resume_against_all_jobs(resume_text_raw, df, top_n=10):
    """
    Full pipeline: parse resume, score against all jobs, return top N results.

    Returns a dict with:
        - top_jobs: list of top N job matches with scores and skill analysis
        - all_results: all jobs sorted by score
        - overall_ats_score: single resume-level ATS score
        - manipulation: detection result
        - sections_detected: which resume sections were found
    """
    print("  [1/4] Parsing resume sections...")
    sections = parse_resume_sections(resume_text_raw)
    weighted_resume = sections_to_weighted_text(sections)
    resume_preprocessed = preprocess_text(weighted_resume)
    resume_full = preprocess_text(resume_text_raw)

    print("  [2/4] Detecting keyword manipulation...")
    is_manipulated, penalty, flags = detect_manipulation(resume_full)
    if is_manipulated:
        print(f"  ⚠️  Manipulation detected! Penalty: -{penalty} pts")
    else:
        print("  ✅ No manipulation detected")

    print(f"  [3/4] Scoring against {len(df)} jobs (TF-IDF + Semantic)...")
    results = []
    for idx, row in df.iterrows():
        job_skills_text = preprocess_text(row["Skills_Required"])
        job_skills_list = row["Skills_List"]

        tfidf_score    = compute_tfidf_score(resume_preprocessed, job_skills_text)
        semantic_score = compute_semantic_score(resume_preprocessed, job_skills_text)
        hybrid_raw     = compute_hybrid_score(tfidf_score, semantic_score)
        final_score    = max(0.0, hybrid_raw - penalty)

        matched, missing = match_skills(resume_text_raw, job_skills_list)
        match_pct = len(matched) / len(job_skills_list) * 100 if job_skills_list else 0

        results.append({
            "job_id":                row["Job_ID"],
            "job_title":             row["Job_Title"],
            "experience_level":      row["Experience_Level"],
            "min_experience_years":  row["Min_Experience_Years"],
            "ats_score":             round(final_score, 2),
            "tfidf_score":           round(tfidf_score * 100, 2),
            "semantic_score":        round(semantic_score * 100, 2),
            "skill_match_pct":       round(match_pct, 2),
            "matched_skills":        matched,
            "missing_skills":        missing,
            "total_required_skills": len(job_skills_list),
        })

    # ── Step 4: Deduplicate — keep best score per unique job title ──────────
    print("  [4/4] Deduplicating results and computing overall score...")
    best_per_title = {}
    for r in results:
        title = r["job_title"]
        if title not in best_per_title or r["ats_score"] > best_per_title[title]["ats_score"]:
            best_per_title[title] = r

    results_sorted = sorted(best_per_title.values(), key=lambda x: x["ats_score"], reverse=True)

    # ── Overall Resume ATS Score: weighted avg of top 5 unique job scores ───
    top5_scores = [r["ats_score"] for r in results_sorted[:5]]
    weights     = [0.35, 0.25, 0.20, 0.12, 0.08]
    # use only as many weights as there are scores (in case fewer than 5 titles)
    weights     = weights[:len(top5_scores)]
    weight_sum  = sum(weights)
    overall_ats_score = round(
        sum(s * w for s, w in zip(top5_scores, weights)) / weight_sum, 2
    )

    print(f"  ✅ Done! Overall ATS Score: {overall_ats_score:.1f} / 100")

    return {
        "top_jobs":           results_sorted[:top_n],
        "all_results":        results_sorted,
        "overall_ats_score":  overall_ats_score,
        "manipulation":       {"detected": is_manipulated, "penalty": penalty, "flags": flags},
        "sections_detected":  {k: bool(v) for k, v in sections.items()}
    }

print("✅ Hybrid scorer and job matching engine defined")

✅ Hybrid scorer and job matching engine defined


## 💡 Step 12 — Recommendation Engine
> Generates personalised, actionable advice based on analysis results.

In [46]:
def generate_recommendations(analysis_result, resume_text_raw):
    """
    Generate actionable resume improvement recommendations.
    Covers: manipulation, missing sections, skill gaps, abbreviations, length.
    """
    recommendations = []
    top_jobs  = analysis_result["top_jobs"]
    sections  = analysis_result["sections_detected"]
    manip     = analysis_result["manipulation"]
    word_count = len(resume_text_raw.split())

    # 1. Manipulation warning
    if manip["detected"]:
        flag_text = "; ".join(manip["flags"][:3])
        recommendations.append({
            "category": "⚠️  Keyword Stuffing Detected",
            "detail":   (f"A penalty of {manip['penalty']:.1f} pts was applied. "
                         f"Flagged terms: {flag_text}. "
                         "Write naturally and avoid repeating the same keywords excessively."),
            "severity": "high"
        })

    # 2. Missing sections
    for sec, present in sections.items():
        if sec == "other":
            continue
        if not present:
            tips = {
                "skills":     "Add a 'Technical Skills' or 'Core Competencies' section listing your tools and technologies.",
                "experience": "Add a 'Work Experience' section with your job roles, company names, and key achievements.",
                "projects":   "Add a 'Projects' section describing 2-3 relevant projects with technologies used.",
                "education":  "Add an 'Education' section with your degree, institution, and year of completion."
            }
            recommendations.append({
                "category": f"📂 Missing Section: {sec.title()}",
                "detail":   tips.get(sec, f"Add a '{sec.title()}' section to your resume."),
                "severity": "medium"
            })

    # 3. Skill gaps for top 3 jobs
    seen_skills = set()
    for job in top_jobs[:3]:
        priority_missing = [s for s in job["missing_skills"] if s not in seen_skills][:5]
        if priority_missing:
            seen_skills.update(priority_missing)
            recommendations.append({
                "category": f"🎯 Add Skills for {job['job_title']} (Score: {job['ats_score']:.1f})",
                "detail":   (f"Your resume is missing: {', '.join(priority_missing)}. "
                             f"Adding these would improve your match for '{job['job_title']}' roles. "
                             "Only add skills you genuinely possess."),
                "severity": "high"
            })

    # 4. Abbreviation tip
    abbrev_check = {
        r"\bnlp\b": "Natural Language Processing (NLP)",
        r"\bml\b":  "Machine Learning (ML)",
        r"\bai\b":  "Artificial Intelligence (AI)",
        r"\bcv\b":  "Computer Vision (CV)",
        r"\baws\b": "Amazon Web Services (AWS)"
    }
    found_abbrevs = []
    for pattern, full_form in abbrev_check.items():
        if re.search(pattern, resume_text_raw.lower()):
            found_abbrevs.append(full_form)
    if found_abbrevs:
        recommendations.append({
            "category": "🔤 Expand Abbreviations",
            "detail":   (f"You used abbreviations: {', '.join(found_abbrevs[:3])}. "
                         "Write them out in full at least once so ATS systems can match both forms."),
            "severity": "low"
        })

    # 5. Resume length
    if word_count < 200:
        recommendations.append({
            "category": "📝 Resume Too Short",
            "detail":   (f"Your resume is only ~{word_count} words. "
                         "Aim for 400–900 words. Add more detail to experience, projects, and skills sections."),
            "severity": "high"
        })
    elif word_count > 1200:
        recommendations.append({
            "category": "📝 Resume May Be Too Long",
            "detail":   (f"Your resume is ~{word_count} words. "
                         "Consider trimming to 600–900 words. Focus on most relevant experience."),
            "severity": "low"
        })

    # 6. ATS score context
    best_score = top_jobs[0]["ats_score"] if top_jobs else 0
    if best_score >= 70:
        recommendations.append({
            "category": "🌟 Strong ATS Match",
            "detail":   (f"Your best score is {best_score:.1f}/100 — excellent! "
                         "Focus on tailoring your resume to the specific job description and adding any missing key skills."),
            "severity": "low"
        })
    elif best_score >= 45:
        recommendations.append({
            "category": "📈 Moderate ATS Match",
            "detail":   (f"Your best score is {best_score:.1f}/100. "
                         "Add the missing skills listed above and use terminology that matches job descriptions more closely."),
            "severity": "medium"
        })
    else:
        recommendations.append({
            "category": "📉 Low ATS Match",
            "detail":   (f"Your best score is only {best_score:.1f}/100. "
                         "Your resume may be in a different career domain than the dataset, or needs significant rework. "
                         "Focus on matching job description keywords exactly."),
            "severity": "high"
        })

    return recommendations

print("✅ Recommendation engine defined")

✅ Recommendation engine defined


## 🖥️ Step 13 — Report Display Functions
> Renders a rich HTML report in Colab.

In [47]:
def score_color(score):
    if score >= 70: return "#16a34a"
    if score >= 45: return "#d97706"
    return "#dc2626"

def severity_style(sev):
    return {
        "high":   "background:#fef2f2;border-left:4px solid #ef4444;",
        "medium": "background:#fffbeb;border-left:4px solid #f59e0b;",
        "low":    "background:#eff6ff;border-left:4px solid #3b82f6;"
    }.get(sev, "")

def display_full_report(analysis_result, recommendations):
    """Render full ATS report as HTML in Colab."""
    top_jobs = analysis_result["top_jobs"]
    manip    = analysis_result["manipulation"]
    sections = analysis_result["sections_detected"]
    best     = top_jobs[0] if top_jobs else {}

    sc            = best.get("ats_score", 0)
    scc           = score_color(sc)
    overall       = analysis_result.get("overall_ats_score", 0)   # ← NEW
    overall_color = score_color(overall)                            # ← NEW

    css = """
<style>
  .ats { font-family: 'Segoe UI', Arial, sans-serif; max-width: 920px; margin: auto; color: #1e293b; }
  .ats-hdr { background: linear-gradient(135deg,#0f172a,#1d4ed8); color:#fff; padding:28px 24px;
             border-radius:14px; text-align:center; margin-bottom:18px; }
  .ats-hdr h1 { margin:0; font-size:1.9em; letter-spacing:-0.5px; }
  .ats-hdr p  { margin:6px 0 0; opacity:.8; font-size:.95em; }
  .card { background:#fff; border:1px solid #e2e8f0; border-radius:12px;
          padding:18px 20px; margin-bottom:16px; box-shadow:0 2px 8px rgba(0,0,0,.06); }
  .card h3 { margin:0 0 14px; color:#0f172a; border-bottom:2px solid #f1f5f9;
             padding-bottom:10px; font-size:1.1em; }
  .big-score { font-size:3.2em; font-weight:800; line-height:1; }
  .meta      { color:#64748b; font-size:.92em; margin-top:6px; }
  .two-col   { display:flex; gap:16px; margin-bottom:16px; }
  .two-col .card { flex:1; margin-bottom:0; text-align:center; }
  .divider   { width:1px; background:#e2e8f0; margin:0 4px; }
  .badge { display:inline-block; padding:4px 12px; border-radius:20px; font-size:.82em; margin:3px; font-weight:600; }
  .b-yes { background:#dcfce7; color:#166534; }
  .b-no  { background:#fee2e2; color:#991b1b; }
  .jrow { background:#f8fafc; border-left:5px solid #2563eb; padding:14px 16px;
          margin-bottom:12px; border-radius:0 10px 10px 0; }
  .jrow h4 { margin:0 0 8px; font-size:1em; color:#0f172a; }
  .jmeta { display:flex; gap:18px; font-size:.87em; color:#374151; flex-wrap:wrap; margin-bottom:6px; }
  .bar-bg   { background:#e2e8f0; border-radius:99px; height:8px; margin:4px 0 10px; }
  .bar-fill { height:8px; border-radius:99px; }
  .stag { display:inline-block; padding:3px 9px; border-radius:16px; font-size:.8em; margin:2px; }
  .s-match { background:#dcfce7; color:#166534; }
  .s-miss  { background:#fee2e2; color:#991b1b; }
  .rec { padding:12px 14px; margin-bottom:9px; border-radius:8px; font-size:.93em; }
  .rec strong { display:block; margin-bottom:3px; font-size:1em; }
  table { width:100%; border-collapse:collapse; font-size:.9em; }
  th { background:#0f172a; color:#fff; padding:10px 12px; text-align:left; }
  td { padding:9px 12px; border-bottom:1px solid #f1f5f9; }
  tr:hover td { background:#f8fafc; }
  .score-chip { font-weight:700; }
  .score-label { font-size:.78em; color:#94a3b8; font-weight:400; margin-top:4px; }
  .score-formula { font-size:.78em; color:#94a3b8; margin-top:6px; font-style:italic; }
</style>
"""

    html = css + '<div class="ats">'

    # ── Header ────────────────────────────────────────────────────────────────
    html += f"""
    <div class="ats-hdr">
      <h1>🎯 Hybrid ATS Resume Analysis Report</h1>
      <p>TF-IDF (40%) + Semantic Similarity (60%) + Section-Aware + Manipulation Detection</p>
    </div>"""

    # ── Overall ATS Score + Best Match (side by side) ─────────────────────── ← NEW BLOCK
    html += f"""
    <div class="two-col">

      <div class="card">
        <h3>📊 Overall Resume ATS Score</h3>
        <div class="big-score" style="color:{overall_color}">
          {overall:.1f}<span style="font-size:.4em;color:#94a3b8"> / 100</span>
        </div>
        <div class="score-label">Weighted avg of top 5 unique role scores</div>
        <div class="score-formula">Formula: (0.4 × TF-IDF + 0.6 × Semantic) × 100 − Penalty</div>
      </div>

      <div class="card">
        <h3>🏆 Best Matching Role</h3>
        <div class="big-score" style="color:{scc}">
          {sc:.1f}<span style="font-size:.4em;color:#94a3b8"> / 100</span>
        </div>
        <div class="meta" style="margin-top:10px">
          <strong>{best.get('job_title','N/A')}</strong><br>
          Level: {best.get('experience_level','N/A')} &nbsp;|&nbsp;
          Min Exp: {best.get('min_experience_years','N/A')} yrs<br>
          Skill Match: {best.get('skill_match_pct',0):.1f}%
          ({len(best.get('matched_skills',[]))}/{best.get('total_required_skills',0)} skills)
        </div>
      </div>

    </div>"""

    # ── Resume Sections ───────────────────────────────────────────────────────
    html += '<div class="card"><h3>📋 Resume Sections Detected</h3>'
    for sec, found in sections.items():
        if sec == "other": continue
        cls = "b-yes" if found else "b-no"
        ico = "✅" if found else "❌"
        html += f'<span class="badge {cls}">{ico} {sec.title()}</span>'
    html += '</div>'

    # ── Manipulation ──────────────────────────────────────────────────────────
    if manip["detected"]:
        flags_html = "<br>".join([f"&nbsp;&nbsp;• {f}" for f in manip["flags"]])
        html += f"""
        <div class="card" style="border-left:5px solid #ef4444">
          <h3>🚨 Keyword Stuffing Detected — Penalty: <span style="color:#dc2626">−{manip['penalty']:.1f} pts</span></h3>
          <p style="margin:0;color:#7f1d1d;font-size:.9em">{flags_html}</p>
        </div>"""
    else:
        html += '<div class="card" style="border-left:5px solid #22c55e"><h3>✅ No Keyword Stuffing Detected</h3></div>'

    # ── Top Matching Jobs ─────────────────────────────────────────────────────
    html += '<div class="card"><h3>💼 Top Matching Jobs</h3>'
    for i, job in enumerate(top_jobs, 1):
        s     = job["ats_score"]
        sc2   = score_color(s)
        bar_w = min(s, 100)
        m_tags = " ".join([f'<span class="stag s-match">{x}</span>' for x in job["matched_skills"][:8]])
        x_tags = " ".join([f'<span class="stag s-miss">{x}</span>'  for x in job["missing_skills"][:6]])
        html += f"""
        <div class="jrow">
          <h4>#{i} {job['job_title']}
            <span style="font-weight:normal;color:#64748b;font-size:.9em">
              ({job['experience_level']}, {job['min_experience_years']}+ yrs)
              &nbsp;—&nbsp; {job['job_id']}
            </span>
          </h4>
          <div class="jmeta">
            <span>🎯 ATS: <strong style="color:{sc2}">{s:.1f}</strong></span>
            <span>📊 TF-IDF: {job['tfidf_score']:.1f}</span>
            <span>🧠 Semantic: {job['semantic_score']:.1f}</span>
            <span>🔧 Skills: {job['skill_match_pct']:.1f}% ({len(job['matched_skills'])}/{job['total_required_skills']})</span>
          </div>
          <div class="bar-bg"><div class="bar-fill" style="width:{bar_w}%;background:{sc2}"></div></div>
          <div><strong style="font-size:.83em">✅ Matched:</strong> {m_tags if m_tags else '<em style="color:#94a3b8">None detected</em>'}</div>
          <div style="margin-top:5px"><strong style="font-size:.83em">❌ Missing:</strong> {x_tags if x_tags else '<em style="color:#94a3b8">None</em>'}</div>
        </div>"""
    html += '</div>'

    # ── Score Comparison Table ────────────────────────────────────────────────
    html += '<div class="card"><h3>📊 Score Comparison Table</h3><table>'
    html += '<tr><th>#</th><th>Job Title</th><th>Level</th><th>ATS Score</th><th>TF-IDF</th><th>Semantic</th><th>Skill Match</th></tr>'
    for i, job in enumerate(top_jobs, 1):
        sc3 = score_color(job["ats_score"])
        html += f"""<tr>
          <td>{i}</td>
          <td>{job['job_title']}</td>
          <td>{job['experience_level']}</td>
          <td><span class="score-chip" style="color:{sc3}">{job['ats_score']:.1f}</span></td>
          <td>{job['tfidf_score']:.1f}</td>
          <td>{job['semantic_score']:.1f}</td>
          <td>{job['skill_match_pct']:.1f}%</td>
        </tr>"""
    html += '</table></div>'

    # ── Recommendations ───────────────────────────────────────────────────────
    if recommendations:
        html += '<div class="card"><h3>🛠️ Recommendations to Improve Your Resume</h3>'
        for rec in recommendations:
            html += f"""
            <div class="rec" style="{severity_style(rec['severity'])}">
              <strong>{rec['category']}</strong>{rec['detail']}
            </div>"""
        html += '</div>'

    html += '</div>'  # close .ats
    display(HTML(html))

print("✅ Report display functions defined")

✅ Report display functions defined


## 🚀 Step 14 — Run the Full Analysis

> **Upload your resume PDF or DOCX below.** The system will:
> 1. Parse and section your resume
> 2. Detect keyword manipulation
> 3. Score against all jobs in the dataset
> 4. Display your full report


In [48]:
# ─── Upload Resume ───────────────────────────────────────────────────────────
print("📎 Please upload your resume (PDF or DOCX format)")
uploaded_resume = files.upload()
resume_filename = list(uploaded_resume.keys())[0]
print(f"✅ Resume uploaded: {resume_filename}")

# ─── Extract Text ─────────────────────────────────────────────────────────────
print("\n⏳ Extracting resume text...")
resume_text_raw = extract_resume_text(resume_filename)
word_count = len(resume_text_raw.split())
print(f"   → Extracted {word_count} words")

if word_count < 50:
    print("⚠️  WARNING: Very little text extracted. If your PDF is image-based,")
    print("   consider converting it to text-based PDF or DOCX first.")

# ─── Run Hybrid Analysis ──────────────────────────────────────────────────────
print("\n⏳ Running hybrid ATS analysis...")
print("  (This may take 30-90 seconds depending on dataset size)")
print("-" * 50)

analysis_result = analyze_resume_against_all_jobs(resume_text_raw, df, top_n=10)

# ─── Generate Recommendations ────────────────────────────────────────────────
recommendations = generate_recommendations(analysis_result, resume_text_raw)

# ─── Display Full HTML Report ─────────────────────────────────────────────────
print("\n" + "="*60)
print("           📋  FULL ATS ANALYSIS REPORT")
print("="*60 + "\n")
display_full_report(analysis_result, recommendations)

# ─── Plain Text Summary ───────────────────────────────────────────────────────
print("\n" + "="*60)
print("           PLAIN TEXT SUMMARY")
print("="*60)

top     = analysis_result["top_jobs"][0]
overall = analysis_result.get("overall_ats_score", 0)          # ← NEW

# Overall score first, then best match breakdown
print(f"\n📊 Overall Resume ATS Score : {overall:.1f} / 100")   # ← NEW
print(f"   (Weighted avg of top 5 unique role scores)")         # ← NEW
print(f"   Formula: (0.4 × TF-IDF + 0.6 × Semantic) × 100 − Penalty") # ← NEW

print(f"\n🏆 Best Matching Role")
print(f"   Job Title  : {top['job_title']}")
print(f"   ATS Score  : {top['ats_score']:.1f} / 100")
print(f"   Level      : {top['experience_level']}")
print(f"   Skill Match: {top['skill_match_pct']:.1f}%  "
      f"({len(top['matched_skills'])}/{top['total_required_skills']} skills)")
print(f"   Penalty    : -{analysis_result['manipulation']['penalty']:.1f} pts")

print(f"\n📋 Top 5 Matching Jobs:")
print(f"   {'#':<4} {'Job Title':<32} {'ATS Score':<12} {'Level'}")
print(f"   {'-'*4} {'-'*32} {'-'*12} {'-'*15}")
for i, j in enumerate(analysis_result["top_jobs"][:5], 1):
    print(f"   {i:<4} {j['job_title']:<32} {j['ats_score']:>5.1f}/100     {j['experience_level']}")

print(f"\n🛠️  Key Recommendations:")
for r in recommendations[:5]:
    print(f"   [{r['severity'].upper():<6}] {r['category']}")

print("\n" + "="*60)

📎 Please upload your resume (PDF or DOCX format)


Saving Palak_Verma_Resume (1).pdf to Palak_Verma_Resume (1) (2).pdf
✅ Resume uploaded: Palak_Verma_Resume (1) (2).pdf

⏳ Extracting resume text...
   → Extracted 349 words

⏳ Running hybrid ATS analysis...
  (This may take 30-90 seconds depending on dataset size)
--------------------------------------------------
  [1/4] Parsing resume sections...
  [2/4] Detecting keyword manipulation...
  ✅ No manipulation detected
  [3/4] Scoring against 74 jobs (TF-IDF + Semantic)...
  [4/4] Deduplicating results and computing overall score...
  ✅ Done! Overall ATS Score: 36.7 / 100

           📋  FULL ATS ANALYSIS REPORT



#,Job Title,Level,ATS Score,TF-IDF,Semantic,Skill Match
1,Full Stack Developer,All Levels,37.9,7.4,58.3,19.2%
2,Analytics Engineer,All Levels,37.3,4.8,58.9,12.5%
3,Customer Success Engineer,All Levels,36.4,5.4,57.0,13.3%
4,Data Architect,All Levels,34.1,4.3,54.0,8.3%
5,Data Scientist,All Levels,33.9,9.5,50.2,30.8%
6,Backend Developer,All Levels,33.7,6.5,51.9,15.4%
7,Software Engineer,All Levels,33.3,7.8,50.3,28.6%
8,Data Engineer,All Levels,33.3,5.2,52.1,16.0%
9,AI Engineer,All Levels,32.7,4.0,51.8,8.0%
10,Data Analyst,All Levels,31.8,4.2,50.1,12.5%



           PLAIN TEXT SUMMARY

📊 Overall Resume ATS Score : 36.7 / 100
   (Weighted avg of top 5 unique role scores)
   Formula: (0.4 × TF-IDF + 0.6 × Semantic) × 100 − Penalty

🏆 Best Matching Role
   Job Title  : Full Stack Developer
   ATS Score  : 37.9 / 100
   Level      : All Levels
   Skill Match: 19.2%  (5/26 skills)
   Penalty    : -0.0 pts

📋 Top 5 Matching Jobs:
   #    Job Title                        ATS Score    Level
   ---- -------------------------------- ------------ ---------------
   1    Full Stack Developer              37.9/100     All Levels
   2    Analytics Engineer                37.3/100     All Levels
   3    Customer Success Engineer         36.4/100     All Levels
   4    Data Architect                    34.1/100     All Levels
   5    Data Scientist                    33.9/100     All Levels

🛠️  Key Recommendations:
   [HIGH  ] 🎯 Add Skills for Full Stack Developer (Score: 37.9)
   [HIGH  ] 🎯 Add Skills for Analytics Engineer (Score: 37.3)
   [HIGH  ]

## 💾 Step 15 (Optional) — Export Full Results to JSON

In [49]:
import json
_files_placeholder = None

export = {
    "best_match": {
        "job_title":        analysis_result["top_jobs"][0]["job_title"],
        "ats_score":        analysis_result["top_jobs"][0]["ats_score"],
        "experience_level": analysis_result["top_jobs"][0]["experience_level"],
        "skill_match_pct":  analysis_result["top_jobs"][0]["skill_match_pct"],
    },
    "top_10_jobs": [
        {
            "rank":             i + 1,
            "job_id":           j["job_id"],
            "job_title":        j["job_title"],
            "experience_level": j["experience_level"],
            "ats_score":        j["ats_score"],
            "tfidf_score":      j["tfidf_score"],
            "semantic_score":   j["semantic_score"],
            "skill_match_pct":  j["skill_match_pct"],
            "matched_skills":   j["matched_skills"],
            "missing_skills":   j["missing_skills"],
        }
        for i, j in enumerate(analysis_result["top_jobs"])
    ],
    "manipulation_check":  analysis_result["manipulation"],
    "sections_detected":   analysis_result["sections_detected"],
    "recommendations":     recommendations,
}

with open("ats_analysis_result.json", "w") as f:
    json.dump(export, f, indent=2)

files.download("ats_analysis_result.json")
print("✅ Results exported and downloaded as ats_analysis_result.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Results exported and downloaded as ats_analysis_result.json
